# 第十三章 智能旅行助手

在前面的章节中，我们从零开始构建了 HelloAgents 框架，实现了多种智能体范式、工具系统、记忆机制、协议通信和性能评估等核心功能。从本章开始，我们将进入一个全新的阶段：**将所学知识融会贯通，构建完整的实用应用。**

还记得在第一章中，我们构建的第一个智能体吗？那是一个简单的智能旅行助手，展示了 `Thought-Action-Observation` 循环的基本原理。本章的智能旅行助手将是一个完整的项目，包含以下核心功能：

**（1）智能行程规划**：用户输入目的地、日期、偏好等信息，系统自动生成包含景点、餐饮、酒店的完整行程计划。

**（2）地图可视化**：在地图上标注景点位置、绘制游览路线，让行程一目了然。

**（3）预算计算**：自动计算门票、酒店、餐饮、交通费用，显示预算明细。

**（4）行程编辑**：支持添加、删除、调整景点，实时更新地图。

**（5）导出功能**：支持导出为 PDF 或图片，方便保存和分享。

## 13.1 项目概述与架构设计

### 13.1.1 为什么需要智能旅行助手

规划一次旅行是一件既令人兴奋又令人头疼的事情。你需要在网上搜索景点信息，对比不同的攻略，查看天气预报，预订酒店，计算预算，规划路线。这个过程可能需要花费几个小时甚至几天的时间。而且即使花了这么多时间，你也不确定规划的行程是否合理，是否遗漏了什么重要的景点，预算是否准确。

传统的旅行规划方式有几个痛点：

- **信息分散**：景点信息在旅游网站上，天气信息在天气网站上，酒店信息在预订网站上，你需要在多个网站之间切换，手动整合这些信息。
- **缺少个性化**：大部分攻略都是通用的，不考虑你的个人偏好、预算限制、出行时间等因素。
- **难以调整**：当你想修改行程时，可能需要重新规划整个行程。

AI 技术为解决这些问题提供了新的可能。想象一下，你只需要告诉系统
3
，系统就能自动为你生成一个完整的行程计划，包括每天去哪些景点、在哪里吃饭、住哪个酒店、需要多少预算。

### 13.1.2 技术架构概览

系统采用经典的**前后端分离架构**，分为四个层次（图 13.1）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/13-figures/13-1.png" alt="" width="85%"/>
  <p>图 13.1 智能旅行助手技术架构</p>
</div>

**（1）前端层 (Vue3+TypeScript)**：负责用户交互和数据展示，包括表单输入、结果展示、地图可视化。

**（2）后端层 (FastAPI)**：负责 API 路由、数据验证、业务逻辑。

**（3）智能体层 (HelloAgents)**：负责任务分解、工具调用、结果整合。包含 4 个专门的 Agent。

**（4）外部服务层**：提供数据和能力，包括高德地图 API、Unsplash API、LLM API。

数据流转过程如下：用户在前端填写表单 → 后端验证数据 → 调用智能体系统 → 智能体依次调用景点搜索、天气查询、酒店推荐、行程规划 Agent → 每个 Agent 通过 MCP 协议调用外部 API → 整合结果返回前端 → 前端渲染展示。

项目的结构参考如下，便于定位源码：

In [ ]:
# 项目目录结构
project_structure = """
helloagents-trip-planner/
├── backend/                    # 后端代码
│   ├── app/
│   │   ├── agents/            # 智能体实现
│   │   ├── api/               # API路由
│   │   ├── models/            # 数据模型
│   │   ├── services/          # 服务层
│   │   └── config.py          # 配置文件
│   └── requirements.txt       # Python依赖
│
└── frontend/                   # 前端代码
    ├── src/
    │   ├── views/             # 页面组件
    │   ├── services/          # API服务
    │   ├── types/             # 类型定义
    │   └── router/            # 路由配置
    └── package.json           # npm依赖
"""
print(project_structure)

### 13.1.3 快速体验：5 分钟运行项目

在深入学习实现细节之前，让我们先把项目跑起来，看看最终的效果。

**环境要求：**
- Python 3.10 或更高版本
- Node.js 16.0 或更高版本
- npm 8.0 或更高版本

**获取 API 密钥：**
- LLM 的 API（OpenAI、DeepSeek 等）
- 高德地图 Web 服务 Key：访问 https://console.amap.com/ 注册并创建应用
- Unsplash Access Key：访问 https://unsplash.com/developers 注册并创建应用

将所有 API 密钥放入 `.env` 文件。

In [ ]:
# 启动后端（在终端中执行）
startup_backend = """
# 1. 进入后端目录
cd helloagents-trip-planner/backend

# 2. 安装依赖
pip install -r requirements.txt

# 3. 配置环境变量
cp .env.example .env
# 编辑 .env 文件，填入你的 API 密钥

# 4. 启动后端服务
uvicorn app.api.main:app --reload
# 或者
python run.py
"""

# 启动前端（在新终端中执行）
startup_frontend = """
# 1. 进入前端目录
cd helloagents-trip-planner/frontend

# 2. 安装依赖
npm install

# 3. 启动前端服务
npm run dev
"""

print("后端启动命令:", startup_backend)
print("前端启动命令:", startup_frontend)
print("\n成功启动后：")
print("  后端 API 文档：http://localhost:8000/docs")
print("  前端应用：    http://localhost:5173")

成功启动后，访问 http://localhost:5173 即可使用应用。

在首页表单中填写目的地城市、旅行日期、偏好、预算、交通及住宿类型等信息，点击
按钮后，系统会显示加载进度条，并很快生成结果页面（图 13.2—13.5）。

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/13-figures/13-2.png" alt="" width="85%"/>
  <p>图 13.2 旅行助手规划进行页面</p>
</div>

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/13-figures/13-3.png" alt="" width="85%"/>
  <p>图 13.3 旅行助手规划完成页面</p>
</div>

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/13-figures/13-4.png" alt="" width="85%"/>
  <p>图 13.4 旅行助手规划完成页面</p>
</div>

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/13-figures/13-5.png" alt="" width="85%"/>
  <p>图 13.5 旅行助手规划完成页面（编辑与导出）</p>
</div>

## 13.2 数据模型设计

### 13.2.1 Web 应用中的数据流转

在构建智能旅行助手时，我们需要解决一个核心问题：**如何表示和传递旅行计划数据？**

数据经历了多次转换：

```
前端表单 → HTTP 请求 → 后端 Python 对象 → 外部 API 响应
        → 后端 Python 对象 → HTTP 响应 → 前端 TypeScript 对象 → 页面展示
```

如果没有统一的数据格式，每一步转换都可能出错。这就是为什么我们需要**数据模型**。

### 13.2.2 从字典到 Pydantic 模型

在第一章的简单原型中，我们使用 Python 字典来表示景点数据，这会带来字段名不统一、类型不安全、维护性差等问题。Pydantic 提供了更好的解决方案。

In [ ]:
# 第一章的做法：使用字典
attraction_dict = {
    "name": "故宫",
    "location": {"lng": 116.397128, "lat": 39.916527},
    "price": 60
}

# 访问数据（没有类型检查，容易出错）
lng = attraction_dict["location"]["lng"]
print(f"第一章的方式 - 经度: {lng}")
print(f"类型: {type(lng)}")

In [ ]:
# Pydantic 方式：使用类定义
from pydantic import BaseModel, Field

class Location(BaseModel):
    longitude: float = Field(..., description="经度")
    latitude: float = Field(..., description="纬度")

'''
类型安全：longitude 和 latitude 被声明为 float，传入字符串或整数会自动转换，传入无法转换的值会抛出验证错误。

Field(...) 表示必填：...（Ellipsis）是 Python 的省略号对象，在 Pydantic 中代表该字段没有默认值、必须提供，否则会报错。

自动文档生成：description 参数会出现在 FastAPI 自动生成的 OpenAPI 文档（docs）中，说明字段含义。

序列化/反序列化：可以直接用 .model_dump() 转为字典，或从字典/JSON 构建对象。
'''

class Attraction(BaseModel):
    name: str
    location: Location
    ticket_price: int = 0

# 创建对象（类型安全）
attraction = Attraction(
    name="故宫",
    location=Location(longitude=116.397128, latitude=39.916527),
    ticket_price=60
)

# 类型安全的访问（IDE 提供代码补全）
lng = attraction.location.longitude
print(f"Pydantic 方式 - 经度: {lng}")
print(f"景点名称: {attraction.name}")
print(f"门票价格: {attraction.ticket_price} 元")

### 13.2.3 Pydantic 的核心概念

在深入设计数据模型之前，让我们先了解 Pydantic 的几个核心概念：字段定义、验证规则和自定义验证器。

In [1]:
from pydantic import BaseModel, Field
from typing import Optional, List

# 字段定义示例
class AttractionDemo(BaseModel):
    name: str = Field(..., description="景点名称")      # 必填
    rating: float = Field(default=0.0, ge=0, le=5)     # 默认值，范围验证,ge=0 表示 rating 必须大于或等于 0，le=5 表示 rating 必须小于或等于 5
    visit_duration: int = Field(default=60, gt=0)       # 大于 0
    description: Optional[str] = None                   # 可选字段,默认值为 None

# 嵌套模型示例
class Hotel(BaseModel):
    name: str
    price_per_night: int = 0

class DayPlanDemo(BaseModel):
    date: str
    attractions: List[AttractionDemo]  # 景点列表
    hotel: Optional[Hotel] = None      # 可选的酒店信息

# 测试验证
try:
    a = AttractionDemo(name="故宫", rating=6.0)  # rating 超出范围，会报错
except Exception as e:
    print(f"验证错误（预期）: {e}")

a = AttractionDemo(name="故宫", rating=4.8, visit_duration=120)
print(f"正确创建: {a.model_dump()}")

验证错误（预期）: 1 validation error for AttractionDemo
rating
  Input should be less than or equal to 5 [type=less_than_equal, input_value=6.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal
正确创建: {'name': '故宫', 'rating': 4.8, 'visit_duration': 120, 'description': None}


In [ ]:
from pydantic import field_validator

# 自定义验证器示例
class WeatherInfo(BaseModel):
    temperature: int
    #在 Pydantic 做任何处理之前，先用这个类方法对 temperature 的输入值做一次清洗转换。
    @field_validator('temperature', mode='before')
    @classmethod
    def parse_temperature(cls, v):
        """解析温度字符串："16°C" -> 16"""
        if isinstance(v, str):
            v = v.replace('°C', '').replace('℃', '').strip()
            return int(v)
        return v

# 测试自动转换
w1 = WeatherInfo(temperature="16°C")  # 字符串自动转换为整数
w2 = WeatherInfo(temperature=20)       # 整数直接使用

print(f"字符串输入 '16°C' -> {w1.temperature} (类型: {type(w1.temperature).__name__})")
print(f"整数输入    20   -> {w2.temperature} (类型: {type(w2.temperature).__name__})")

字符串输入 '16°C' -> 16 (类型: int)
整数输入    20   -> 20 (类型: int)


### 13.2.4 自底向上的模型设计

好的设计原则是**自底向上**：先定义最基础的模型，然后逐步组合成复杂的结构。这样每个模型都很简单，容易理解和维护。

层次关系如下：
```
Location（位置）
    ↓
Attraction（景点）、Meal（餐饮）、Hotel（酒店）
    ↓
DayPlan（单日行程）+ WeatherInfo（天气）+ Budget（预算）
    ↓
TripPlan（完整旅行计划）
```

In [3]:
from pydantic import BaseModel, Field, field_validator
from typing import Optional, List

# ===== 基础层 =====

class Location(BaseModel):
    """位置信息（经纬度坐标）"""
    longitude: float = Field(..., description="经度", ge=-180, le=180)
    latitude: float = Field(..., description="纬度", ge=-90, le=90)


# ===== 实体层 =====

class Attraction(BaseModel):
    """景点信息"""
    name: str = Field(..., description="景点名称")
    address: str = Field(..., description="地址")
    location: Location = Field(..., description="经纬度坐标")
    visit_duration: int = Field(..., description="建议游览时间（分钟）", gt=0)
    description: str = Field(..., description="景点描述")
    category: Optional[str] = Field(default="景点", description="景点类别")
    rating: Optional[float] = Field(default=None, ge=0, le=5, description="评分")
    image_url: Optional[str] = Field(default=None, description="图片URL")
    ticket_price: int = Field(default=0, ge=0, description="门票价格（元）")


class Meal(BaseModel):
    """餐饮信息"""
    type: str = Field(..., description="餐饮类型：breakfast/lunch/dinner/snack")
    name: str = Field(..., description="餐饮名称")
    address: Optional[str] = Field(default=None, description="地址")
    location: Optional[Location] = Field(default=None, description="经纬度坐标")
    description: Optional[str] = Field(default=None, description="描述")
    estimated_cost: int = Field(default=0, description="预估费用（元）")


class Hotel(BaseModel):
    """酒店信息"""
    name: str = Field(..., description="酒店名称")
    address: str = Field(default="", description="酒店地址")
    location: Optional[Location] = Field(default=None, description="酒店位置")
    price_range: str = Field(default="", description="价格范围")
    rating: str = Field(default="", description="评分")
    distance: str = Field(default="", description="距离景点距离")
    type: str = Field(default="", description="酒店类型")
    estimated_cost: int = Field(default=0, description="预估费用（元/晚）")


class Budget(BaseModel):
    """预算信息"""
    total_attractions: int = Field(default=0, description="景点门票总费用")
    total_hotels: int = Field(default=0, description="酒店总费用")
    total_meals: int = Field(default=0, description="餐饮总费用")
    total_transportation: int = Field(default=0, description="交通总费用")
    total: int = Field(default=0, description="总费用")


# ===== 行程层 =====

class WeatherInfo(BaseModel):
    """天气信息"""
    date: str = Field(..., description="日期")
    day_weather: str = Field(..., description="白天天气")
    night_weather: str = Field(..., description="夜间天气")
    day_temp: int = Field(..., description="白天温度（摄氏度）")
    night_temp: int = Field(..., description="夜间温度（摄氏度）")
    wind_direction: str = Field(..., description="风向")
    wind_power: str = Field(..., description="风力")

    @field_validator('day_temp', 'night_temp', mode='before')
    @classmethod
    def parse_temperature(cls, v):
        """解析温度字符串："16°C" -> 16"""
        if isinstance(v, str):
            v = v.replace('°C', '').replace('℃', '').replace('°', '').strip()
            try:
                return int(v)
            except ValueError:
                return 0  # 容错处理
        return v


class DayPlan(BaseModel):
    """单日行程"""
    date: str = Field(..., description="日期")
    day_index: int = Field(..., description="第几天（从0开始）")
    description: str = Field(..., description="当日行程描述")
    transportation: str = Field(..., description="交通方式")
    accommodation: str = Field(..., description="住宿安排")
    hotel: Optional[Hotel] = Field(default=None, description="酒店信息")
    attractions: List[Attraction] = Field(default_factory=list, description="景点列表")
    meals: List[Meal] = Field(default_factory=list, description="餐饮安排")


# ===== 顶层模型 =====

class TripPlan(BaseModel):
    """旅行计划"""
    city: str = Field(..., description="目的地城市")
    start_date: str = Field(..., description="开始日期")
    end_date: str = Field(..., description="结束日期")
    days: List[DayPlan] = Field(default_factory=list, description="每日行程")
    weather_info: List[WeatherInfo] = Field(default_factory=list, description="天气信息")
    overall_suggestions: str = Field(..., description="总体建议")
    budget: Optional[Budget] = Field(default=None, description="预算信息")


print("数据模型定义完成！")
print("模型层次：Location → Attraction/Meal/Hotel/Budget → DayPlan/WeatherInfo → TripPlan")

数据模型定义完成！
模型层次：Location → Attraction/Meal/Hotel/Budget → DayPlan/WeatherInfo → TripPlan


### 13.2.5 数据模型在 Web 应用中的应用

在 FastAPI 中，Pydantic 模型可以直接用作请求和响应的类型定义。FastAPI 会自动进行数据验证、序列化和文档生成。

In [ ]:
# FastAPI 路由示例（不可直接运行，需要在后端项目中使用）
fastapi_example = '''
from fastapi import FastAPI
from app.models.schemas import TripPlanRequest, TripPlan

app = FastAPI()

@app.post("/api/trip/plan", response_model=TripPlan)
async def create_trip_plan(request: TripPlanRequest) -> TripPlan:
    """
    创建旅行计划

    FastAPI 自动：
    1. 验证请求数据 (TripPlanRequest)
    2. 验证响应数据 (TripPlan)
    3. 生成 OpenAPI 文档
    """
    trip_plan = await generate_trip_plan(request)
    return trip_plan
'''

# 对应的 TypeScript 类型定义
typescript_types = '''
// frontend/src/types/index.ts
interface Location {
  longitude: number;
  latitude: number;
}

interface Attraction {
  name: string;
  address: string;
  location: Location;
  visit_duration: number;
  ticket_price: number;
}

interface TripPlan {
  city: string;
  start_date: string;
  end_date: string;
  days: DayPlan[];
}
'''

print("FastAPI 路由示例:")
print(fastapi_example)
print("\n对应的 TypeScript 类型:")
print(typescript_types)

## 13.3 多智能体协作设计

### 13.3.1 为何需要多智能体

使用单个 Agent 完成旅行规划会遇到两个核心问题：

1. **工具调用的限制**：SimpleAgent 每次 `run()` 调用只能执行一个工具，多次调用间难以传递中间状态。
2. **提示词的复杂度**：让单个 Agent 完成所有任务需要极复杂的提示词，难以维护、容易出错、难以调试。

多 Agent 协作的核心思想：**把复杂任务分解成多个简单任务，让不同的 Agent 各司其职**——就像现实中的旅行社分工合作一样。

### 13.3.2 Agent 角色设计

基于任务分解原则，我们设计了四个专门的 Agent（图 13.6）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/13-figures/13-6.png" alt="" width="85%"/>
  <p>图 13.6 多智能体协作流程</p>
</div>

- **AttractionSearchAgent（景点搜索专家）**：根据用户偏好搜索景点信息
- **WeatherQueryAgent（天气查询专家）**：查询目的地天气预报
- **HotelAgent（酒店推荐专家）**：搜索符合住宿需求的酒店
- **PlannerAgent（行程规划专家）**：整合以上信息，生成完整旅行计划（JSON 格式）

In [4]:
# 各 Agent 的提示词设计

ATTRACTION_AGENT_PROMPT = """你是景点搜索专家。

**工具调用格式:**
`[TOOL_CALL:amap_maps_text_search:keywords=景点,city=城市名]`

**示例:**
- `[TOOL_CALL:amap_maps_text_search:keywords=景点,city=北京]`
- `[TOOL_CALL:amap_maps_text_search:keywords=博物馆,city=上海]`

**重要:**
- 必须使用工具搜索，不要编造信息
- 根据用户偏好({preferences})搜索{city}的景点
"""

WEATHER_AGENT_PROMPT = """你是天气查询专家。

**工具调用格式:**
`[TOOL_CALL:amap_maps_weather:city=城市名]`

请查询{city}的天气信息。
"""

HOTEL_AGENT_PROMPT = """你是酒店推荐专家。

**工具调用格式:**
`[TOOL_CALL:amap_maps_text_search:keywords=酒店,city=城市名]`

请搜索{city}的{accommodation}酒店。
"""

PLANNER_AGENT_PROMPT = """你是行程规划专家。

**输出格式:**
严格按照以下JSON格式返回:
{
  "city": "城市名称",
  "start_date": "YYYY-MM-DD",
  "end_date": "YYYY-MM-DD",
  "days": [...],
  "weather_info": [...],
  "overall_suggestions": "总体建议",
  "budget": {...}
}

**规划要求:**
1. weather_info必须包含每天的天气
2. 温度为纯数字(不带°C)
3. 每天安排2-3个景点
4. 考虑景点距离和游览时间
5. 包含早中晚三餐
6. 提供实用建议
7. 包含预算信息
"""

print("Agent 提示词定义完成！")
print(f"  景点Agent提示词长度: {len(ATTRACTION_AGENT_PROMPT)} 字符")
print(f"  天气Agent提示词长度: {len(WEATHER_AGENT_PROMPT)} 字符")
print(f"  酒店Agent提示词长度: {len(HOTEL_AGENT_PROMPT)} 字符")
print(f"  规划Agent提示词长度: {len(PLANNER_AGENT_PROMPT)} 字符")

Agent 提示词定义完成！
  景点Agent提示词长度: 268 字符
  天气Agent提示词长度: 81 字符
  酒店Agent提示词长度: 110 字符
  规划Agent提示词长度: 322 字符


### 13.3.3 Agent 协作流程

四个 Agent 按以下顺序协作完成旅行规划任务：

In [ ]:
# TripPlannerAgent 协调者实现（伪代码，展示协作流程）
class TripPlannerAgent:
    def __init__(self):
        # 实际项目中需要传入 llm 实例
        # from hello_agents import SimpleAgent, HelloAgentsLLM
        # self.llm = HelloAgentsLLM()
        # self.attraction_agent = SimpleAgent(name="景点搜索", prompt=ATTRACTION_AGENT_PROMPT)
        # ...
        pass

    def plan_trip(self, request: dict) -> dict:
        """协作完成旅行规划，顺序执行四个步骤"""

        # 步骤 1：景点搜索
        print(f"[步骤1] 景点搜索Agent: 搜索{request['city']}的{request['preferences']}景点...")
        # attraction_response = self.attraction_agent.run(
        #     f"请搜索{request['city']}的{request['preferences']}景点"
        # )
        attraction_response = "[模拟] 找到3个景点: 故宫、天坛、颐和园"

        # 步骤 2：天气查询
        print(f"[步骤2] 天气Agent: 查询{request['city']}的天气...")
        # weather_response = self.weather_agent.run(f"请查询{request['city']}的天气")
        weather_response = "[模拟] 天气良好，未来3天晴"

        # 步骤 3：酒店推荐
        print(f"[步骤3] 酒店Agent: 搜索{request['city']}的{request['accommodation']}酒店...")
        # hotel_response = self.hotel_agent.run(
        #     f"请搜索{request['city']}的{request['accommodation']}酒店"
        # )
        hotel_response = "[模拟] 找到2家酒店: 北京饭店、如家快捷"

        # 步骤 4：整合生成计划
        print(f"[步骤4] 规划Agent: 整合信息，生成完整行程...")
        planner_query = self._build_planner_query(
            request, attraction_response, weather_response, hotel_response
        )
        # planner_response = self.planner_agent.run(planner_query)

        # 步骤 5：解析 JSON
        print("[步骤5] 解析JSON，返回TripPlan对象...")
        # trip_plan = self._parse_trip_plan(planner_response)
        # return trip_plan

        return {"status": "流程演示完成", "query_length": len(planner_query)}

    def _build_planner_query(self, request, attraction_response, weather_response, hotel_response):
        """构建规划 Agent 的查询，整合所有信息"""
        return f"""
请根据以下信息生成{request['city']}的{request.get('days', 3)}日旅行计划:

**用户需求:**
- 目的地: {request['city']}
- 日期: {request['start_date']} 至 {request['end_date']}
- 天数: {request.get('days', 3)}天
- 偏好: {request['preferences']}
- 预算: {request['budget']}
- 交通方式: {request['transportation']}
- 住宿类型: {request['accommodation']}

**景点信息:**
{attraction_response}

**天气信息:**
{weather_response}

**酒店信息:**
{hotel_response}

请生成详细的旅行计划，包括每天的景点安排、餐饮推荐、住宿信息和预算明细。
"""


# 演示协作流程
agent = TripPlannerAgent()
result = agent.plan_trip({
    "city": "北京",
    "start_date": "2024-05-01",
    "end_date": "2024-05-03",
    "days": 3,
    "preferences": "历史文化",
    "budget": "中等",
    "transportation": "公共交通",
    "accommodation": "经济型"
})
print(f"\n最终结果: {result}")

## 13.4 MCP 工具集成详解

### 13.4.1 为什么不直接调用 API

直接调用高德地图 HTTP API 会导致：
- **Agent 无法自主调用**：失去智能决策能力
- **参数传递复杂**：需要在提示词中详细说明十几个参数
- **响应解析困难**：需要处理复杂的 JSON 结构
- **工具管理混乱**：十几个 API 需要手动注册管理

### 13.4.2 高德地图 MCP 集成

MCP（Model Context Protocol）是 Anthropic 提出的标准化协议，用于连接 LLM 和外部工具。我们使用 `amap-mcp-server` 这个 Node.js 实现的 MCP 服务器。

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/13-figures/13-7.png" alt="" width="85%"/>
  <p>图 13.7 amap-mcp-server 工具</p>
</div>

<div align="center">
  <p>表 13.1 高德地图 MCP 工具分类</p>
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/13-figures/13-table-1.png" alt="" width="85%"/>
</div>

> **注意：** 本文档中部分示例使用 `npx` 启动 MCP 服务。而在本节代码仓中，我们实际采用的是 `uvx` 方式。`npx` 面向 JavaScript/Node.js（包来自 npm），`uvx` 面向 Python（包来自 PyPI），两种方式并无优劣之分，请按需选择。

In [ ]:
# MCP 工具集成示例（需要在后端项目中运行）
mcp_integration_code = '''
from hello_agents.tools import MCPTool
from app.config import get_settings

settings = get_settings()

# 创建 MCP 工具
mcp_tool = MCPTool(
    name="amap_mcp",
    command="npx",
    args=["-y", "@sugarforever/amap-mcp-server"],
    env={"AMAP_API_KEY": settings.amap_api_key},
    auto_expand=True  # 自动展开为多个独立工具
)

# auto_expand=True 的效果：一个 MCPTool → 16 个工具！
agent.add_tool(mcp_tool)
print(list(agent.tools.keys()))
# ['amap_maps_text_search', 'amap_maps_weather', ...]
'''

print("MCP 工具集成代码（后端项目中使用）:")
print(mcp_integration_code)

MCP 工具调用的完整流程（图 13.8）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/13-figures/13-8.png" alt="" width="85%"/>
  <p>图 13.8 MCP 工具调用流程</p>
</div>

1. Agent 生成工具调用标记：`[TOOL_CALL:amap_maps_text_search:keywords=景点,city=北京]`
2. HelloAgents 框架解析标记，调用对应 Tool 对象
3. MCPTool 通过 JSON-RPC 发送请求给 MCP 服务器进程
4. MCP 服务器调用高德地图 HTTP API
5. 解析响应并返回文本内容给 Agent

In [ ]:
# MCP 协议消息格式示例
import json

# 请求消息（HelloAgents → MCP 服务器）
mcp_request = {
    "jsonrpc": "2.0",
    "method": "tools/call",
    "params": {
        "name": "amap_maps_text_search",
        "arguments": {
            "keywords": "景点",
            "city": "北京"
        }
    }
}

# 响应消息（MCP 服务器 → HelloAgents）
mcp_response = {
    "jsonrpc": "2.0",
    "result": {
        "content": [
            {
                "type": "text",
                "text": "找到以下景点：\n1. 故宫博物院 - 地址：东城区景山前街4号\n2. 天坛公园 - 地址：东城区天坛路\n..."
            }
        ]
    }
}

print("MCP 请求消息:")
print(json.dumps(mcp_request, ensure_ascii=False, indent=2))
print("\nMCP 响应消息:")
print(json.dumps(mcp_response, ensure_ascii=False, indent=2))

MCP 请求消息:
{
 "jsonrpc": "2.0",
 "method": "tools/call",
 "params": {
  "name": "amap_maps_text_search",
  "arguments": {
   "keywords": "景点",
   "city": "北京"
  }
 }
}

MCP 响应消息:
{
  "jsonrpc": "2.0",
  "result": {
    "content": [
      {
        "type": "text",
        "text": "找到以下景点：\n1. 故宫博物院 - 地址：东城区景山前街4号\n2. 天坛公园 - 地址：东城区天坛路\n..."
      }
    ]
  }
}


### 13.4.3 共享 MCP 实例

在多 Agent 系统中，三个 Agent 都需要使用高德地图工具。更好的做法是让所有 Agent **共享同一个 MCPTool 实例**，只启动一个 MCP 服务器进程，节省资源并更好地控制 API 调用频率。

In [ ]:
# 共享 MCP 实例的实现（后端项目中使用）
shared_mcp_code = '''
class TripPlannerAgent:
    def __init__(self):
        settings = get_settings()
        self.llm = HelloAgentsLLM()

        # 创建共享的 MCP 工具实例（只创建一次，只启动一个进程）
        self.mcp_tool = MCPTool(
            name="amap_mcp",
            command="npx",
            args=["-y", "@sugarforever/amap-mcp-server"],
            env={"AMAP_API_KEY": settings.amap_api_key},
            auto_expand=True
        )

        # 创建多个 Agent，共享同一个 MCP 工具
        self.attraction_agent = SimpleAgent(
            name="AttractionSearchAgent",
            llm=self.llm,
            system_prompt=ATTRACTION_AGENT_PROMPT
        )
        self.attraction_agent.add_tool(self.mcp_tool)  # 共享

        self.weather_agent = SimpleAgent(
            name="WeatherQueryAgent",
            llm=self.llm,
            system_prompt=WEATHER_AGENT_PROMPT
        )
        self.weather_agent.add_tool(self.mcp_tool)  # 共享

        self.hotel_agent = SimpleAgent(
            name="HotelAgent",
            llm=self.llm,
            system_prompt=HOTEL_AGENT_PROMPT
        )
        self.hotel_agent.add_tool(self.mcp_tool)  # 共享
'''

print("共享 MCP 实例代码:")
print(shared_mcp_code)
print("优势：3个Agent共享1个MCP服务器进程，节省资源，避免API速率限制")

### 13.4.4 Unsplash 图片 API 集成

我们使用 Unsplash API 为景点获取图片，让旅行计划更加生动直观。注意 Unsplash 是国外服务，搜索结果可能不够准确；实际项目中可考虑使用必应、百度或高德的 POI 图片 API。

Unsplash API 不需要封装成 MCP 工具，因为图片搜索不需要 Agent 的智能决策，只是一个简单的数据增强步骤。

In [ ]:
# backend/app/services/unsplash_service.py
import requests
from typing import Optional, List, Dict
import logging

logger = logging.getLogger(__name__)


class UnsplashService:
    """Unsplash 图片服务"""

    def __init__(self, access_key: str):
        self.access_key = access_key
        self.base_url = "https://api.unsplash.com"

    def search_photos(self, query: str, per_page: int = 10) -> List[Dict]:
        """搜索图片"""
        try:
            url = f"{self.base_url}/search/photos"
            params = {
                "query": query,
                "per_page": per_page,
                "client_id": self.access_key
            }

            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()

            data = response.json()
            results = data.get("results", [])

            # 提取图片 URL
            photos = []
            for result in results:
                photos.append({
                    "url": result["urls"]["regular"],
                    "description": result.get("description", ""),
                    "photographer": result["user"]["name"]
                })

            return photos

        except Exception as e:
            logger.error(f"搜索图片失败: {e}")
            return []

    def get_photo_url(self, query: str) -> Optional[str]:
        """获取单张图片 URL"""
        photos = self.search_photos(query, per_page=1)
        return photos[0].get("url") if photos else None


# 使用示例（需要有效的 access_key）
print("UnsplashService 类定义完成")
print("在 API 路由中使用：")
usage_example = '''
unsplash_service = UnsplashService(settings.unsplash_access_key)

@router.post("/plan", response_model=TripPlan)
async def create_trip_plan(request: TripPlanRequest) -> TripPlan:
    trip_plan = trip_planner_agent.plan_trip(request)

    # 为每个景点获取图片
    for day in trip_plan.days:
        for attraction in day.attractions:
            if not attraction.image_url:
                image_url = unsplash_service.get_photo_url(
                    f"{attraction.name} {trip_plan.city}"
                )
                attraction.image_url = image_url

    return trip_plan
'''
print(usage_example)

## 13.5 前端开发详解

### 13.5.1 前后端分离的 Web 架构

我们的前端采用 Vue 3 + TypeScript 构建单页应用（SPA），通过 HTTP 请求调用后端 API。前端技术栈如下（表 13.2）：

<div align="center">
  <p>表 13.2 前端技术栈</p>
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/13-figures/13-table-2.png" alt="" width="85%"/>
</div>

前端目录结构：
```
frontend/
├── src/
│   ├── views/              # 页面组件
│   │   ├── Home.vue        # 首页（表单）
│   │   └── Result.vue      # 结果页
│   ├── services/           # API 服务
│   │   └── api.ts
│   ├── types/              # 类型定义
│   │   └── index.ts
│   ├── router/             # 路由配置
│   │   └── index.ts
│   ├── App.vue
│   └── main.ts
├── package.json
├── vite.config.ts
└── tsconfig.json
```

### 13.5.2 TypeScript 类型定义

前端的 TypeScript 类型与后端 Pydantic 模型完全对应，确保前后端数据格式统一。

In [ ]:
# frontend/src/types/index.ts （TypeScript 代码，此处以字符串展示）
typescript_types = '''
// frontend/src/types/index.ts

export interface Location {
  longitude: number
  latitude: number
}

export interface Attraction {
  name: string
  address: string
  location: Location
  visit_duration: number
  description: string
  category?: string
  rating?: number
  image_url?: string
  ticket_price?: number
}

export interface Meal {
  type: string
  name: string
  address?: string
  location?: Location
  description?: string
  estimated_cost?: number
}

export interface Hotel {
  name: string
  address: string
  location?: Location
  price_range?: string
  rating?: string
  estimated_cost?: number
}

export interface Budget {
  total_attractions: number
  total_hotels: number
  total_meals: number
  total_transportation: number
  total: number
}

export interface WeatherInfo {
  date: string
  day_weather: string
  night_weather: string
  day_temp: number
  night_temp: number
  wind_direction: string
  wind_power: string
}

export interface DayPlan {
  date: string
  day_index: number
  description: string
  transportation: string
  accommodation: string
  hotel?: Hotel
  attractions: Attraction[]
  meals: Meal[]
}

export interface TripPlan {
  city: string
  start_date: string
  end_date: string
  days: DayPlan[]
  weather_info: WeatherInfo[]
  overall_suggestions: string
  budget?: Budget
}

export interface TripPlanRequest {
  city: string
  start_date: string
  end_date: string
  days: number
  preferences: string
  budget: string
  transportation: string
  accommodation: string
}
'''

print("TypeScript 类型定义（frontend/src/types/index.ts）:")
print(typescript_types)

### 13.5.3 API 服务封装

使用 Axios 封装 HTTP 请求，并添加请求/响应拦截器。超时时间设置为 2 分钟，因为生成旅行计划需要多轮 LLM 调用。

In [ ]:
# frontend/src/services/api.ts（TypeScript 代码）

import axios from 'axios'
import type { TripPlanRequest, TripPlan } from '../types'

const api = axios.create({
  baseURL: 'http://localhost:8000/api',
  timeout: 120000, // 2 分钟超时（生成计划需要多轮 LLM 调用）
  headers: {
    'Content-Type': 'application/json' #HTTP 请求头，告诉服务器我们发送的是 JSON 数据
  }
})

// 请求拦截器：只是打印日志，方便调试
api.interceptors.request.use(
  config => {
    console.log('发送请求：', config)  // 能看到每次请求的 URL、参数
    return config
  },
  error => Promise.reject(error)  // 请求构建失败（极少发生）
)

// 响应拦截器：打印响应，统一处理错误
api.interceptors.response.use(
  response => {
    console.log('收到响应：', response)  // 能看到每次响应的数据
    return response
  },
  error => {
    console.error('请求失败：', error)  // 所有请求的失败都在这里统一捕获
    return Promise.reject(error)        // 继续把错误往外抛，让调用方处理
  }
)

// 生成旅行计划（前端调用后端的唯一入口）
export const generateTripPlan = async (request: TripPlanRequest): Promise<TripPlan> => {
  const response = await api.post<TripPlan>('/trip/plan', request)
  return response.data
}

print("API 服务封装代码（frontend/src/services/api.ts）:")
print(api_service_code)

### 13.5.4 Home 表单设计

Home 页面使用 Vue 3 Composition API 组织代码，包含表单数据绑定、加载状态管理和进度条模拟。

In [ ]:
# Home.vue 核心逻辑（TypeScript 部分）
home_vue_script = '''
<script setup lang="ts">
import { ref } from 'vue'
import { useRouter } from 'vue-router'
import { message } from 'ant-design-vue'
import { generateTripPlan } from '@/services/api'
import type { TripPlanRequest } from '@/types'

const router = useRouter()
const loading = ref(false)
const loadingProgress = ref(0)
const loadingStatus = ref('')

// 表单数据（双向绑定）
const formData = ref<TripPlanRequest>({
  city: '',
  start_date: '',
  end_date: '',
  days: 3,
  preferences: '历史文化',
  budget: '中等',
  transportation: '公共交通',
  accommodation: '经济型酒店'
})

const handleSubmit = async () => {
  loading.value = true
  loadingProgress.value = 0

  // 模拟进度更新（实际进度无法从后端获取）
  const progressInterval = setInterval(() => {
    if (loadingProgress.value < 90) {
      loadingProgress.value += 10
      if (loadingProgress.value <= 30) loadingStatus.value = '🔍 正在搜索景点...'
      else if (loadingProgress.value <= 50) loadingStatus.value = '🌤️ 正在查询天气...'
      else if (loadingProgress.value <= 70) loadingStatus.value = '🏨 正在推荐酒店...'
      else loadingStatus.value = '📋 正在生成行程计划...'
    }
  }, 500)

  try {
    const response = await generateTripPlan(formData.value)
    clearInterval(progressInterval)
    loadingProgress.value = 100
    router.push({ name: 'result', state: { tripPlan: response } })
  } catch (error) {
    clearInterval(progressInterval)
    message.error('生成计划失败，请重试')
  } finally {
    loading.value = false
  }
}
</script>
'''

print("Home.vue 脚本部分:")
print(home_vue_script)

In [ ]:
# Home.vue 模板部分（使用 Ant Design Vue 组件）
home_vue_template = '''
<template>
  <div class="home-container">
    <div class="page-header">
      <h1 class="page-title">✈️ 智能旅行助手</h1>
      <p class="page-subtitle">基于AI的个性化旅行规划</p>
    </div>

    <a-card class="form-card">
      <a-form :model="formData" @finish="handleSubmit">
        <a-form-item label="目的地城市" name="city" :rules="[{ required: true }]">
          <a-input v-model:value="formData.city" placeholder="如：北京" />
        </a-form-item>

        <!-- 更多表单项... -->

        <a-form-item>
          <a-button type="primary" html-type="submit" size="large" :loading="loading">
            开始规划
          </a-button>
        </a-form-item>

        <!-- 加载进度条 -->
        <a-form-item v-if="loading">
          <a-progress :percent="loadingProgress" status="active" />
          <p>{{ loadingStatus }}</p>
        </a-form-item>
      </a-form>
    </a-card>
  </div>
</template>
'''

print("Home.vue 模板部分:")
print(home_vue_template)

### 13.5.5 Result 页面展示

Result 页面展示生成的旅行计划，包含：行程概览、预算明细、地图可视化（高德地图 JS API）、每日行程详情、天气信息。

In [ ]:
# Result.vue 地图初始化代码（TypeScript）
result_map_code = '''
import AMapLoader from '@amap/amap-jsapi-loader'

const initMap = async () => {
  const AMap = await AMapLoader.load({
    key: 'your_amap_web_key',
    version: '2.0'
  })

  map = new AMap.Map('amap-container', {
    zoom: 12,
    center: [116.397128, 39.916527]
  })

  // 添加景点标记
  tripPlan.value.days.forEach((day) => {
    day.attractions.forEach((attraction, index) => {
      const marker = new AMap.Marker({
        position: [attraction.location.longitude, attraction.location.latitude],
        title: attraction.name,
        label: { content: `${index + 1}`, direction: 'top' }
      })
      map.add(marker)
    })
  })
}
'''

# 导出功能代码
export_code = '''
import html2canvas from 'html2canvas'
import jsPDF from 'jspdf'

// 导出为图片
const exportAsImage = async () => {
  const element = document.getElementById('trip-plan-content')
  const canvas = await html2canvas(element, { scale: 2, useCORS: true })
  const link = document.createElement('a')
  link.download = `${tripPlan.value.city}旅行计划.png`
  link.href = canvas.toDataURL()
  link.click()
}

// 导出为 PDF
const exportAsPDF = async () => {
  const element = document.getElementById('trip-plan-content')
  const canvas = await html2canvas(element, { scale: 2, useCORS: true })
  const imgData = canvas.toDataURL('image/png')
  const pdf = new jsPDF('p', 'mm', 'a4')
  const imgWidth = 210  // A4 宽度
  const imgHeight = (canvas.height * imgWidth) / canvas.width
  pdf.addImage(imgData, 'PNG', 0, 0, imgWidth, imgHeight)
  pdf.save(`${tripPlan.value.city}旅行计划.pdf`)
}
'''

print("地图初始化代码:")
print(result_map_code)
print("\n导出功能代码:")
print(export_code)

## 13.6 功能实现详解

本节介绍智能旅行助手的核心功能实现，包括预算计算、加载进度条、行程编辑、导出功能和侧边导航。

### 13.6.1 预算计算功能

预算计算逻辑在后端 PlannerAgent 中实现，LLM 根据行程中的景点、酒店、餐饮安排估算每一项费用。在前端，使用 Ant Design Vue 的 `Statistic` 组件展示预算明细。

In [ ]:
# 预算展示组件（Vue 模板）
budget_template = '''
<a-card v-if="tripPlan.budget" title="💰 预算明细">
  <a-row :gutter="16">
    <a-col :span="6">
      <a-statistic title="景点门票" :value="tripPlan.budget.total_attractions" suffix="元" />
    </a-col>
    <a-col :span="6">
      <a-statistic title="酒店住宿" :value="tripPlan.budget.total_hotels" suffix="元" />
    </a-col>
    <a-col :span="6">
      <a-statistic title="餐饮费用" :value="tripPlan.budget.total_meals" suffix="元" />
    </a-col>
    <a-col :span="6">
      <a-statistic title="交通费用" :value="tripPlan.budget.total_transportation" suffix="元" />
    </a-col>
  </a-row>
  <a-divider />
  <a-row>
    <a-col :span="24" style="text-align: center;">
      <a-statistic
        title="预估总费用"
        :value="tripPlan.budget.total"
        suffix="元"
        :value-style="{ color: '#cf1322', fontSize: '32px', fontWeight: 'bold' }"
      />
    </a-col>
  </a-row>
</a-card>
'''

# 用 Python 模拟预算计算逻辑
def calculate_budget(day_plans: list) -> dict:
    """模拟预算计算（实际由 LLM 生成）"""
    total_attractions = sum(
        attraction.get('ticket_price', 0)
        for day in day_plans
        for attraction in day.get('attractions', [])
    )
    total_hotels = sum(
        day.get('hotel', {}).get('estimated_cost', 0)
        for day in day_plans
        if day.get('hotel')
    )
    total_meals = sum(
        meal.get('estimated_cost', 0)
        for day in day_plans
        for meal in day.get('meals', [])
    )
    total_transportation = 200  # 估算交通费

    return {
        'total_attractions': total_attractions,
        'total_hotels': total_hotels,
        'total_meals': total_meals,
        'total_transportation': total_transportation,
        'total': total_attractions + total_hotels + total_meals + total_transportation
    }

# 测试
mock_days = [
    {'attractions': [{'ticket_price': 60}, {'ticket_price': 15}],
     'hotel': {'estimated_cost': 300},
     'meals': [{'estimated_cost': 50}, {'estimated_cost': 80}, {'estimated_cost': 60}]},
    {'attractions': [{'ticket_price': 30}],
     'hotel': {'estimated_cost': 300},
     'meals': [{'estimated_cost': 50}, {'estimated_cost': 70}, {'estimated_cost': 60}]}
]

budget = calculate_budget(mock_days)
print("预算计算结果:")
for key, value in budget.items():
    print(f"  {key}: {value} 元")

### 13.6.2 加载进度条

生成旅行计划需要 10-30 秒，加载进度条（模拟进度）能有效提升用户体验，让用户知道系统正在工作。（已在 13.5.4 节的 `handleSubmit` 函数中实现）

### 13.6.3 行程编辑功能

编辑功能的核心是**状态管理**：维护当前行程和原始行程两个状态，支持撤销修改。

In [ ]:
# 行程编辑功能（TypeScript）
edit_code = '''
const editMode = ref(false)
const originalPlan = ref<TripPlan | null>(null)

// 进入编辑模式（深拷贝保存原始计划）
const toggleEditMode = () => {
  editMode.value = true
  originalPlan.value = JSON.parse(JSON.stringify(tripPlan.value))  // 深拷贝
}

// 移动景点（交换数组元素）
const moveAttraction = (dayIndex: number, attractionIndex: number, direction: 'up' | 'down') => {
  const attractions = tripPlan.value.days[dayIndex].attractions
  const newIndex = direction === 'up' ? attractionIndex - 1 : attractionIndex + 1

  if (newIndex >= 0 && newIndex < attractions.length) {
    // ES6 解构赋值交换，无需临时变量
    [attractions[attractionIndex], attractions[newIndex]] =
    [attractions[newIndex], attractions[attractionIndex]]
  }
}

// 删除景点
const deleteAttraction = (dayIndex: number, attractionIndex: number) => {
  tripPlan.value.days[dayIndex].attractions.splice(attractionIndex, 1)
}

// 保存修改（重新初始化地图）
const saveChanges = () => {
  editMode.value = false
  message.success('修改已保存')
  initMap()  // 地图随行程变化而更新
}

// 取消编辑（恢复原始计划）
const cancelEdit = () => {
  if (originalPlan.value) {
    tripPlan.value = originalPlan.value
  }
  editMode.value = false
}
'''

print("行程编辑功能代码:")
print(edit_code)

# Python 模拟：数组元素交换
attractions = ["故宫", "天坛", "颐和园"]
print("\n交换前:", attractions)
idx, direction = 0, 'down'
new_idx = idx + 1 if direction == 'down' else idx - 1
if 0 <= new_idx < len(attractions):
    attractions[idx], attractions[new_idx] = attractions[new_idx], attractions[idx]
print("交换后:", attractions)

### 13.6.4 导出功能

提供导出为图片和 PDF 两种方式，使用 `html2canvas` 和 `jsPDF` 库。

> **技术难点**：地图是 Canvas 渲染的，`html2canvas` 处理嵌套 Canvas 存在兼容性问题。当前采用简化方案（导出时隐藏地图），实际项目可考虑：使用高德静态地图 API、服务端 Puppeteer 截图等替代方案。

（详细代码已在 13.5.5 节中展示）

### 13.6.5 侧边导航与锚点跳转

Result 页面内容较多，提供侧边导航实现平滑滚动定位。

In [ ]:
# 侧边导航与锚点跳转（TypeScript）
nav_code = '''
const activeSection = ref('overview')

// 滚动到指定区域（浏览器原生 API）
const scrollToSection = ({ key }: { key: string }) => {
  activeSection.value = key
  const element = document.getElementById(key)
  if (element) {
    element.scrollIntoView({ behavior: 'smooth', block: 'start' })
    //                        ^^^^^^^^^^^ 平滑滚动  ^^^^^^^^^^^^^ 顶部对齐
  }
}
'''

nav_template = '''
<a-menu
  v-model:selectedKeys="[activeSection]"
  mode="inline"
  @click="scrollToSection"
>
  <a-menu-item key="overview">📋 行程概览</a-menu-item>
  <a-menu-item key="budget">💰 预算明细</a-menu-item>
  <a-menu-item key="map">🗺️ 地图</a-menu-item>
  <a-menu-item key="days">📅 每日行程</a-menu-item>
  <a-menu-item key="weather">🌤️ 天气</a-menu-item>
</a-menu>
'''

print("侧边导航逻辑:")
print(nav_code)
print("侧边导航模板:")
print(nav_template)

## 13.7 结语

恭喜你完成了第十三章的学习！

通过本章，你不仅学会了如何构建一个完整的智能旅行助手应用，更重要的是掌握了：

1. **系统设计思维**：如何将复杂问题分解为多个简单任务
2. **工程实践能力**：如何将理论知识转化为可运行的代码
3. **全栈开发能力**：如何整合前后端技术栈
4. **AI 应用开发**：如何利用 LLM 构建实用的应用

这个项目是一个起点，而不是终点。你可以基于这个项目：

- 添加更多功能（餐厅推荐 Agent、交通规划 Agent 等）
- 优化用户体验（WebSocket 实时进度推送等）
- 扩展到其他领域（智能购物助手、智能学习助手等）
- 部署到生产环境，服务真实用户

最好的学习方式是实践。不要只是阅读代码，而是要动手修改、扩展、优化。每一次实践都会让你对多 Agent 系统有更深的理解。

**祝你在 AI 应用开发的道路上越走越远！**